# Project.py

In [1]:
import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings(action='ignore')

In [ ]:
from pathlib import Path
from src.project import PolymerBuildProject, exported_to_lammps, load_job_melt_neat_topology, monomers_allowed


project_path = Path('polyID_test')
# project_path = Path('polyID_production')
project = PolymerBuildProject.get_project(project_path)

In [ ]:
project.print_status(detailed=True)

#### Run Jobs

In [ ]:
project.run()

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        # 'pack_lattice',
        # 'to_interchange',
        'md_export'
    ],
    # jobs=[
    #     project.open_job(id='214e8512b6218c21668ce8c19c6bf359')
    # ]
)

## Inspect jobs

### Checking that aromaticity is being respected

In [ ]:
job = project.open_job(id='ce13320b89716c28e297df8ab26ddc8b')

In [ ]:
from rdkit import Chem

sdf_path = job.fn(PolymerBuildProject.OLIGOMER_SDF)
sdf_path

In [ ]:
with Chem.SDMolSupplier(sdf_path) as supp:
    for mol in supp:
        display(mol)

In [ ]:
from polymerist.rdutils import set_rdkdraw_size, disable_kekulized_drawing

disable_kekulized_drawing()
set_rdkdraw_size(500, 3/2)

display(mol)

In [ ]:
import pandas as pd

records : list[dict] = []
for job in project:
    op_times = job.doc.get(PolymerBuildProject.OP_TIME_RECORD_NAME)
    if op_times is not None:
        op_times['Job ID'] = job.id
        records.append(op_times)

### Inspect multiple jobs

In [ ]:
from polymerist.genutils.decorators.functional import allow_string_paths

@allow_string_paths
def ls(path : Path) -> None:
    if not path.is_dir():
        raise NotADirectoryError
    for subpath in path.iterdir():
        print(subpath.name)

@allow_string_paths
def cat(path : Path) -> None:
    if not path.is_file():
        raise IsADirectoryError
    
    with path.open('r') as file:
        for line in file.readlines():
            print(line, end='')

In [ ]:
notable_jobs = [
    '214e8512b6218c21668ce8c19c6bf359', # stereochemistry is inverted by file write
    'a4b0a6be41f5786ff9f77f777cda59fa', # long alkane backbone hangs up partitions algorithm
    '41bc5b7d9700c043b78c0b898de4e758', # hangs with NAGL charges
]

In [ ]:
job = project.open_job(id=notable_jobs[-1])
project.get_job_status(job)

In [ ]:
offmol = load_job_oligomer_molecule(job)
offmol

In [ ]:
assign_partial_charges(job)

In [ ]:
cat(job.fn(PolymerBuildProject.LOGFILE_NAME))

In [ ]:
cat(job.fn(PolymerBuildProject.OLIGOMER_SDF))

In [ ]:
ls(job.path)

In [ ]:
with open(job.fn(PolymerBuildProject.LOGFILE_NAME), 'r') as file:
    for line in file.readlines():
        print(line)